In [5]:
import os
import json
import urllib.request
import urllib.parse
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from pybliometrics.scopus import ScopusSearch
from pybliometrics.utils import create_config, init

# ==========================================
# 1. SETUP LOCAL DIRECTORY
# ==========================================
# This creates the 'MuTech/01_Raw_Exports' folder wherever you run the script
current_dir = os.getcwd()
folder_path = os.path.join(current_dir, 'MuTech', '01_Raw_Exports')
os.makedirs(folder_path, exist_ok=True)
print(f"Ready to save files locally to: {folder_path}\n")

# ==========================================
# 2. SCOPUS EXTRACTION & AUTOMATED FILTERING
# ==========================================
print("\nStarting Scopus extraction...")
try:
    init() 
except FileNotFoundError:
    print("Scopus config not found. Please run create_config() in your terminal manually.")

try:
    scopus_query = 'TITLE-ABS-KEY(("music" OR "instrument" OR "music information retrieval" OR "audio signal processing") AND ("performance" OR "solo" OR "ensemble" OR "transcription") AND ("machine learning" OR "AI" OR "artificial intelligence" OR "deep learning")) AND PUBYEAR > 2019'
    search_results = ScopusSearch(scopus_query)

    if search_results.results:
        df_scopus = pd.DataFrame(search_results.results)
        initial_count = len(df_scopus)
        
        # Automated Filtering: Keep only empirical studies (drops books/reviews)
        valid_types = ['Article', 'Conference Paper']
        df_scopus = df_scopus[df_scopus['subtypeDescription'].isin(valid_types)]
        
        # Automated Filtering: Drop rows without an abstract or authors
        df_scopus = df_scopus.dropna(subset=['description', 'creator'])
        
        # Pruning and Renaming Columns for Rayyan
        df_scopus = df_scopus[['title', 'description', 'coverDate', 'doi', 'creator', 'authkeywords', 'subtypeDescription']]
        df_scopus = df_scopus.rename(columns={
            'title': 'Title',
            'description': 'Abstract',
            'coverDate': 'Year',
            'doi': 'DOI',
            'creator': 'Authors',
            'authkeywords': 'Keywords',
            'subtypeDescription': 'Document Type'
        })
        
        # Clean the Year format
        df_scopus['Year'] = df_scopus['Year'].astype(str).str[:4]
        df_scopus['Source'] = 'Scopus'

        final_count = len(df_scopus)
        print(f"Total Scopus papers originally extracted: {initial_count}")
        print(f"Total Scopus papers kept after automated exclusion: {final_count}")
        print(f"Junk papers automatically dropped: {initial_count - final_count}")
        
        if not df_scopus.empty:
            df_scopus.to_csv(os.path.join(folder_path, 'Scopus_Filtered.csv'), index=False)
            print("Scopus filtered CSV saved successfully.")
    else:
        print("No Scopus results found or query failed.")
except Exception as e:
    print(f"Scopus extraction failed: {e}")


print("\nAll extractions complete! Check the MuTech/01_Raw_Exports folder.")

Ready to save files locally to: c:\Users\E1560289\Documents\MuTech\MuTech\01_Raw_Exports


Starting Scopus extraction...
Total Scopus papers originally extracted: 9970
Total Scopus papers kept after automated exclusion: 8943
Junk papers automatically dropped: 1027
Scopus filtered CSV saved successfully.

All extractions complete! Check the MuTech/01_Raw_Exports folder.


In [ ]:
import urllib.request
import urllib.parse
import json
import pandas as pd

# ==========================================
# 5. IEEE XPLORE EXTRACTION (UPDATED)
# ==========================================
print("\nStarting IEEE extraction...")
ieee_api_key = "h24uddpgdmamhmxhmgsfhzg3"

try:
    ieee_query = '("music" OR "instrument" OR "music information retrieval" OR "audio signal processing") AND ("performance" OR "solo" OR "ensemble" OR "transcription") AND ("machine learning" OR "AI" OR "artificial intelligence" OR "deep learning")'
    encoded_query = urllib.parse.quote(ieee_query)
    
    # We can use HTTPS here as well for better security
    url = f"https://ieeexploreapi.ieee.org/api/v1/search/articles?apikey={ieee_api_key}&format=json&max_records=200&querytext={encoded_query}"

    # FIX: Add Browser Headers to bypass the 403 Firewall block
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'application/json'
    }

    # Swap urllib for requests to easily pass headers
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        print(f"Total IEEE papers extracted: {data.get('total_records', 0)}")
        
        if 'articles' in data:
            df_ieee = pd.DataFrame(data['articles'])
            df_ieee.to_csv(os.path.join(folder_path, 'IEEE_Raw.csv'), index=False)
            print("IEEE CSV saved successfully.")
        else:
            print("No articles found in the IEEE response.")
    else:
        print(f"IEEE API blocked the request. Status Code: {response.status_code}")
        print(f"Server Response: {response.text}")

except Exception as e:
    print(f"IEEE extraction failed: {e}")

HTTPError: HTTP Error 403: Forbidden